# Tutorial: Agent 编排架构 + LangGraph (Oxford-style)

## Persona (cell 1 - Oxford tutorial fellow)

You are an **Oxford tutorial fellow in Agent 编排架构与 LangGraph**.

- **Never give direct answers.** Use Socratic questioning. Reject vague claims ("差不多" / "大概" / "应该").
- **End each turn with a probing question.** Why? How could? What if? 反例? 凭什么? 依据?
- **Play devil's advocate**: when the student asserts X, argue ~X and demand evidence (Christensen Center HBS method).
- **Scaffold fade**: if defense fails, drop one level (L0 直问 -> L1 给提示 -> L2 给反例 -> L3 给半截 worked example). Never collapse to direct answer.
- Hold the student to 三连: "为什么用图不用链? 凭什么? 反例?"

This notebook simulates a 1-on-1 Oxford tutorial. You are the fellow; the reader is the student. **No direct answers, ever.** (Vygotsky 共构 + 对话式教学法 + Socratic method)


## Pre-tutorial task (强制 retrieval, 不准翻 notes.md)

Before the tutorial begins, you MUST submit (in 100-200 字 each, 不查资料):

1. **画图**: 画一个 `research -> strategy -> copywriter -> approval -> publish` 的状态图, 标出哪里是**顺序 / 条件 / 循环 / HITL**。
2. **塌缩推演**: 若把 `interrupt_before=["approval"]` 去掉, HITL 三步模式会塌缩成几步? 为什么? `update_state` 在没有暂停点时是什么行为?
3. **拓扑辩护**: A2A 与 MCP 各负责什么? 给一个"既要 MCP 又要 A2A"的企业场景, 解释为什么缺一不可。

> **不交 pre-task 不开 tutorial。** 这是 Oxford tutorial 强制 retrieval 的工程化 (测试效应 / 提取练习, Butler 2010: 检索 >> 重学)。
> 提交后写入 `student_model.json` 的 `pre_task` 字段, tutorial 读取后追问。


In [ ]:
# Socratic loop (>=4 轮, 每轮检测 defense 失败则降一级 scaffold, 禁直接答案)
# 静态 if/else 模拟 LLM, 不调 API。每轮必以追问结尾。

# 模拟学员 pre-task 答案 (实际从 student_model.json 读取)
STUDENT_ANSWERS = {
    "q1": "顺序是 research->strategy, 条件是 approval, 循环是 copywriter<->approval, HITL 是 interrupt_before",
    "q2": "塌缩成一步, 因为没有暂停点",
    "q3": "MCP 接工具, A2A 接 Agent",
}

SCAFFOLD = {0: "L0 直问", 1: "L1 给提示", 2: "L2 给反例", 3: "L3 给半截 worked"}

def socratic_turn(qid, ans, rnd):
    # 单轮 Socratic, 追问 + 禁直接答案。返回下一 round。
    print(f"\n--- Round {rnd} ({SCAFFOLD[rnd % 4]}) ---")
    if qid == "q1":
        if "条件" in ans and "interrupt" not in ans.lower():
            # 追问 1: 凭什么 approval 是条件不是循环? 反例?
            print("[Fellow] 你说 approval 是条件分支。**凭什么?** 若 approval 永远 false, 你的图会怎样? **反例**: 这是不是死循环? (Socratic 追问 1)")
        elif "interrupt" in ans.lower():
            # 追问 2: 为什么 interrupt_before 必须配 update_state?
            print("[Fellow] 你提到了 interrupt_before。**为什么** interrupt_before 必须配 update_state? 不配会怎样? **如何**让图恢复执行? (Socratic 追问 2)")
        else:
            print("[Fellow] 你漏了 HITL。重画, 标出 interrupt_before 在哪个节点前。 (Scaffold L1)")
    elif qid == "q2":
        if "一步" in ans:
            # 追问 3: 塌缩成一步? 为什么 update_state 是 no-op?
            print("[Fellow] 你说塌缩成一步。但 invoke(initial) 仍会跑过 approval 节点 (不停)。**那人工决策怎么注入?** update_state 在没有暂停点时是什么行为? **为什么**? (Socratic 追问 3)")
        elif "两步" in ans or "二步" in ans:
            # 追问 4: 两步? 哪两步?
            print("[Fellow] 塌缩成两步? **哪两步?** **假设** revision_count 已 =3, 你的两步能退出循环吗? **凭什么**? (Socratic 追问 4)")
        else:
            print("[Fellow] 重答: HITL 三步模式的每一步分别依赖 interrupt_before 的什么性质? (Scaffold L2)")
    elif qid == "q3":
        if "MCP 接工具" in ans and "A2A 接 Agent" in ans:
            # 追问 5: 反例 - 单 Agent + 多工具需不需要 A2A?
            print("[Fellow] 口诀对了。**反例**: 若我只有一个 Agent + 多个工具, 需要 A2A 吗? **为什么**? **假设**变成 3 个 Agent 跨进程, 没有 A2A 会怎样? (Socratic 追问 5)")
        else:
            print("[Fellow] 口诀不完整。重述: MCP 接___, A2A 接___。 (Scaffold L1)")
    return rnd + 1

# 4+ 轮 Socratic loop (静态模拟, 每问 2 轮: 直问 + devil's advocate)
rnd = 1
for q, a in STUDENT_ANSWERS.items():
    rnd = socratic_turn(q, a, rnd)              # 第 1 轮: 直问
    rnd = socratic_turn(q, "devil: " + a, rnd)  # 第 2 轮: devil's advocate 反驳

print(f"\n[共 {rnd-1} 轮 Socratic 追问完成。禁直接答案 - 学员须自答。]")


In [ ]:
# student_model.json 读写 (跨单元复用, tutorial 与 drill 共享)
import json, os
from datetime import datetime

DEFAULT_MODEL = {
    "unit": "skill2-day2-agent-orchestration-langgraph",
    "mastery": {
        "S1_StateGraph装配": 0.0,   # 0-1, 由 D1 reps 更新 (worked=0.3, faded=0.3, independent=0.4)
        "S2_HITL三步": 0.0,
        "S3_拓扑选型": 0.0,
    },
    "blind_spots": [],              # tutorial exit 时填充, 供下单元读取
    "drill_history": [],            # [{drill_id, stage, passed, ts}]
    "pre_task": None,               # pre-task 答案 (cell 2 提交)
    "last_tutorial": None,          # ISO timestamp, 限频用
    "scaffold_level": "L0",         # L0/L1/L2/L3, Socratic 失败降级
    "weak_loop_count": 0,           # 连续 2 次失败触发
    "review_schedule": "schedule.json",  # 指向 FSRS-6 复习表
    "recommended_review": [],       # 跨单元复习指针 (如技能5 Day2)
}

def load_student_model(path="student_model.json"):
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    return json.loads(json.dumps(DEFAULT_MODEL))  # deep copy

def save_student_model(model, path="student_model.json"):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(model, f, ensure_ascii=False, indent=2)

def update_mastery(model, subskill, delta):
    # D1/D2/D3 完成后调用, 更新掌握度 (0-1 cap)。
    model["mastery"][subskill] = min(1.0, model["mastery"].get(subskill, 0.0) + delta)
    model["drill_history"].append({"ts": datetime.now().isoformat(), "subskill": subskill, "delta": delta})
    return model

def add_blind_spot(model, spot):
    # tutorial exit 时调用, 记录盲点供下单元复用 (Hattie [FEED-FORWARD])。
    if spot not in model["blind_spots"]:
        model["blind_spots"].append(spot)
    return model

def check_rate_limit(model):
    # 限频: 每单元 1 次/天 (与 Oxford tutorial 每周 1 次同构, 防依赖)。
    if model.get("last_tutorial"):
        last = datetime.fromisoformat(model["last_tutorial"])
        if (datetime.now() - last).total_seconds() < 86400:
            return False, "限频: 距上次 tutorial 不足 24h, 明天再来 (强制学员先自己想)"
    return True, "OK"

# 示例: 学员完成 D1 worked + faded 阶段, tutorial 暴露盲点
m = load_student_model()
ok, msg = check_rate_limit(m)
print(f"[限频检查] {msg}")
if ok:
    m = update_mastery(m, "S1_StateGraph装配", 0.6)  # worked + faded
    m = add_blind_spot(m, "条件边 vs 普通边的区别不熟 (tutorial 追问 1 暴露)")
    m = add_blind_spot(m, "A2A 与 MCP 互补场景想不出 (tutorial 追问 5 暴露)")
    m["last_tutorial"] = datetime.now().isoformat()
    m["recommended_review"] = ["技能5 Day2 LangGraph mechanics (StateGraph 基础)"]
    save_student_model(m)
    print("student_model.json 已更新:")
    print(json.dumps(m, ensure_ascii=False, indent=2))


## Hattie (2007) 四级 formative feedback

> 参考: Hattie (2007) *Educational Psychology Review* 77(1):81-112. **避免 Self 级表扬** (实证: Self 级表扬反促学习)。聚焦 TASK / PROCESS / SELF-REG / FEED-FORWARD。

### [TASK] 任务级反馈 (针对具体答案对错)
- 学员 pre-task Q1 漏 HITL: "你的图缺 `interrupt_before` 节点。重画, 在 approval 前加暂停点。"
- 学员 D1 faded 把条件边写成普通 `add_edge`: "这是普通 Edge, 不是 Conditional Edge。用 `add_conditional_edges` + 条件函数。"
- 学员 Q2 答"塌缩成一步": "错。`invoke(initial)` 仍跑过 approval, 只是停不下。人工决策注入不了。"

### [PROCESS] 过程级反馈 (针对策略/方法)
- 学员 D2 漏 `update_state`: "你的 HITL 只有 invoke 没有 update_state。三步模式的**第二步是注入人工决策**, 不是 just `invoke(None)`。**策略错了, 不是代码错了。**"
- 学员 D3 把 Supervisor 混同 Plan-Execute: "Supervisor 是路由角色 (有全局视图), Plan-Execute 是两阶段架构。你的 `strategy_agent` 是 Plan, 不是 Supervisor。**概念混淆。**"
- 学员跳过 worked 直接做 independent: "你跳了 worked 阶段。强制 retrieval 不是赶进度。回去看一遍 worked example 再来。"

### [SELF-REG] 自我调节级反馈 (针对元认知)
- 学员连续 2 次 D2 失败: "你两次都漏 update_state。**停下来**, 用 why-what-how 三问反思: 为什么漏 / 正确的是什么 / 下次怎么自查。这是 self-reg, 不是再跑一次。"
- 学员 weak_loop 触发后仍硬冲: "你已触发 weak_loop。**元认知检查**: 你是真的不懂, 还是没看 worked? 若没看, 先看; 若看了不懂, 写下卡在哪一行。"
- 学员 blind_spots 为空却提交: "你的 blind_spots 是空的。tutorial 没暴露任何盲点 = 你没认真被追问。重做。"

### [FEED-FORWARD] 前馈级反馈 (针对下步)
- 学员 D1 通过: "**下一步做 D2 (HITL)。** 预习 notes.md §关键回顾5。注意 `MemorySaver` vs `SqliteSaver` 的区别 -- 你下个 Day 会用到 `SqliteSaver`。"
- 学员全部通过 (mastery S1/S2/S3 >= 0.8): "**进入 Day 3 (人机协作治理)。** 今天的 `interrupt_before` 是技术实现, Day 3 讲组织层面的治理设计。把今天的 `student_model.json` 带过去, Day 3 会读你的 `blind_spots`。"
- 学员 A2A/MCP 仍混: "**回技能5 Day2** 复习 LangGraph mechanics, 再回来做 D3 independent。schedule.json C3 间隔复习已排。"


## 限频与 Exit (防依赖 + 闭环)

### 限频 (防 LLM 依赖)
- **本 tutorial 每天最多 1 次** (与 Oxford tutorial 每周 1 次同构, 强制学员先自己想, 防依赖 LLM)
- 连续 2 天使用且 mastery 无提升, 触发 `weak_loop` (见 practice.md)
- `student_model.json` 记录 `last_tutorial` 时间戳, `check_rate_limit()` 跨单元读取
- 限频是工程化的"克制" -- Oxford tutorial 之所以有效, 正因为稀疏 (学员被迫在两次 tutorial 间深度思考)

### Exit artifact (闭环交付)
完成 tutorial 后, 在 `student_model.json` 写入:
- `blind_spots`: **2-3 个**本次 tutorial 暴露的盲点 (如"条件边注册机制不熟" / "A2A 与 MCP 互补场景想不出" / "update_state 在无暂停点时是 no-op 不熟")
- `recommended_review`: 推荐复习的单元 (如"技能5 Day2 LangGraph mechanics" / "本 Day schedule.json C3")
- `scaffold_level`: 若仍为 L2/L3, 标记需重做 D1 worked (未独立)

### 闭环校验
- tutorial exit 时若 `mastery` S1/S2/S3 任一 < 0.5, **强制回 practice.md weak_loop** (不进下一单元)
- 若 `blind_spots` 为空, 视为 tutorial 失败 (学员没认真被追问), 重做
- `schedule.json` C1-C4 间隔复习由 FSRS-6 调度 (due [1,3,8,21,60,180] 天), **不依赖 tutorial**, 独立长期保持
- 跨单元: Day 3 tutorial 会读本 Day `student_model.json` 的 `blind_spots`, 衔接治理设计
